Q 1 :

In [ ]:
from selenium import webdriver
from chromedriver_py import binary_path
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup 
import pandas as pd
import time

In [ ]:

def scape_jobs(search_query,pages=3) :

    service = webdriver.ChromeService(executable_path=binary_path)
    driver = webdriver.Chrome(service = service)

    data = []

    for page in range(pages) :
        driver.get(f'https://jobinja.ir/jobs?&filters%5Bjob_categories%5D%5B%5D=&filters%5Bkeywords%5D%5B0%5D={search_query}')
        time.sleep(3)
        
        soup = BeautifulSoup(driver.page_source , 'html.parser')
        jobs = soup.find_all('li' , class_='c-jobListView__item')
        
        print(f' number of all job : {len(jobs)}')

        for job in jobs :
            title = job.find('a' , class_='c-jobListView__titleLink').text.strip()
            company = job.find_all('li' , class_='c-jobListView__metaItem')[0].text.strip()
            location = job.find_all('li' , class_='c-jobListView__metaItem')[1].text.strip()

            data.append({
                'title' : title ,
                 'company' : company ,
                  'location' : location})
            
        next_button = driver.find_element(By.CSS_SELECTOR , "a[rel='next']")
        driver.execute_script('arguments[0].scrollIntoView();' , next_button)
        time.sleep(3)

        next_button.click()

    driver.quit()
    pd.DataFrame(data).to_csv('jobs.csv' , index=False , encoding='utf-8-sig')

    return data

scape_jobs('برنامه نویس پایتون')

Q 2 :

In [ ]:
import re
from hazm import Normalizer , word_tokenize , stopwords_list , WordTokenizer
import pandas as pd

In [ ]:
df = pd.read_csv("comment.csv")
df.head()

In [ ]:
df.info()

In [ ]:
CONTROL_CHARS_RE = re.compile(r"[\u200c\u200d\u200e\u200f\u202a-\u202e\ufeff\u00ad\u2060\u180e]")
DIGIT_MAP = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
KEEP_CHARS_RE = re.compile(r"[^0-9A-Za-z\u0600-\u06FF\s]")
MULTI_SPACE = re.compile(r"\s+")
PATTERN = re.compile(r"(\S)\1{2,}")

hazm_normalizer = Normalizer(
        correct_spacing=True,
        remove_diacritics=True,
        remove_specials_chars=False,
        decrease_repeated_chars=True,
        persian_style=True,
        persian_numbers=False
    )

def clean_persian_text(text: str) -> str:

    if not text:
        return ""

    t = CONTROL_CHARS_RE.sub(" ", text)
    t = t.translate(DIGIT_MAP)
    t = KEEP_CHARS_RE.sub(" ", t)
    t = MULTI_SPACE.sub(" ", t).strip()
    t = re.sub(PATTERN , r"\1", t)

    replacements = {
        "ي": "ی",
        "ك": "ک",
        "أ": "ا",
        "إ": "ا",
        "ؤ": "و",
        "ة": "ه",
        "ئ": "ی",
    }
    for old, new in replacements.items():
        t = t.replace(old, new)

    hazm_normalize = hazm_normalizer.normalize(t)
    tokenizer = word_tokenize(hazm_normalize)

    return ' '.join(tokenizer)


comment = df['comment'][5]
text_clean = clean_persian_text(comment)
print('befor clean : ' , comment)
print('after clean regex : ' , text_clean)

In [ ]:
df['clean_comment'] = df['comment'].apply(clean_persian_text)
print('after clean : \n' , df['clean_comment'] )

In [ ]:
df.head()

In [ ]:
standard_stopwords = set(stopwords_list())
print(standard_stopwords)

In [ ]:
negation_words = {"نه", "نخیر", "نیست", "نمی", "نمي", "نبود", "نشد", "بدون"}
final_stopwords = standard_stopwords - negation_words
tokenizer = WordTokenizer(separate_emoji=False, replace_links=False, replace_ids=False)

def second_cleand(text) :
    if not text: return ''

    tokens = tokenizer.tokenize(text)
    standard = [t for t in tokens if len(t) >= 2 and t not in standard_stopwords]
    return ' '.join(standard)

print("Running Step 3 (This might take a minute)...")
df['stopwords'] = df['comment'].apply(second_cleand)

In [ ]:
df.head()

In [ ]:
from collections import Counter

def most_common_word(dataframe , column_name , top=10) :

    all_word = []
    for text in dataframe[column_name] :
        all_word.extend(word_tokenize(text))

    word_count = Counter(all_word)

    most_common_df = pd.DataFrame(word_count.most_common(top) , columns=['repeat','word'])

    return most_common_df


print(' 10 most common word : \n' , most_common_word(df , 'stopwords' , top=10))

Q 3 :

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [ ]:
def recommendation_movie(movie_title , df) :

    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(df['overview'].fillna(''))

    idx = df[df['title'] == movie_title ].index[0]
    cosine_simi = cosine_similarity(tfidf_matrix[idx],tfidf_matrix)

    sim_scores = list(enumerate(cosine_simi[0]))
    sim_scores = sorted(sim_scores , key=lambda x:x[1] , reverse=True)

    movie_indices = [i[0] for i in sim_scores[1:6]]

    return df['title'].iloc[movie_indices]

In [ ]:
df1 = pd.read_csv('tmdb_5000_movies.csv')
new_df = df1[['title' , 'overview']] 
new_df.head()

In [ ]:
print( recommendation_movie('The Dark Knight' , new_df) )

Q 4 :

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report , confusion_matrix ,accuracy_score
from sklearn.svm import LinearSVC
import seaborn as sns
import matplotlib.pyplot as plt
import re
import numpy as np
import pandas as pd

In [ ]:
df2 = pd.read_csv('asriran.csv')
df2.head()

In [ ]:
def preprocess_text(text) :

    text = clean_persian_text(text)
    text = second_cleand(text)

    return text

text = df2['body'][5]
print('befor clean : ' , text)
print('cleaned text : ' , preprocess_text(text))

In [ ]:
df2['body'] = df2['body'].fillna('')
df2['clean_body'] = df2['body'].apply(preprocess_text)
print('after clean : \n ' , df2['clean_body'] )

In [ ]:
df.info()

In [ ]:
x = df2['clean_body']
y = df2['subgroup']

x_train , x_test , y_train , y_test = train_test_split(
    x , y , test_size=0.2 , random_state=42 , stratify=y
)

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1,2) ,
    max_features=50000 ,
    min_df=2
)

x_train_tfidf = vectorizer.fit_transform(x_train)
x_test_tfidf = vectorizer.transform(x_test)

In [ ]:
model = LogisticRegression(max_iter=2000 , class_weight='balanced')

model.fit(x_train_tfidf , y_train)

In [ ]:
y_pred = model.predict(x_test_tfidf)

acc = accuracy_score(y_test , y_pred)

print( f'accuracy : {acc :.4f} ')
print('classification report : \n ' , classification_report(y_test,y_pred))

In [ ]:
labels = np.unique(y)
cm = confusion_matrix(y_test , y_pred , labels=labels)

plt.figure(figsize=(10 , 6))
sns.heatmap(
    cm ,
    annot=False,
    cmap='Blues' ,
    xticklabels=labels,
    yticklabels=labels
)
plt.title('confusion matrix')
plt.xlabel('predicted label')
plt.ylabel('true lable')
plt.xticks(rotation=45 , ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
error_per_class = {}

for i,label in enumerate(labels) :
    total_true = cm[i].sum()
    correct = cm[i,i]
    errors = total_true - correct
    error_per_class[label]= errors

top2_misclassified = sorted(error_per_class.items() , key=lambda x:x[1] , reverse=True)[:2]

print('top 2 most misclassified classes :')
for cls ,err in top2_misclassified :
    print(f'{cls} : {err} misclassified samples')

In [ ]:
for cls ,err in top2_misclassified :
    idx = np.where(labels == cls)[0][0]
    row = cm[idx].copy()
    row[idx] = 0
    confused_with_idx = row.argmax()
    confused_with_label = label[confused_with_idx]

print(f'class {cls} is most often confused with {confused_with_label} ' )

In [ ]:
miss_idx = np.where(y_test != y_pred)[0]
print(f'total misclassified sample : {len(miss_idx)}')

for i in miss_idx[:5] :
    print('-'*50)
    print('text : ' , x_test.iloc[i])
    print('true : ' , y_test.iloc[i] , '| pred :' , y_pred[i])